# DemandVision AI — Exploratory Data Analysis (EDA)
## Kaggle Store Item Demand Forecasting Dataset

This notebook performs an in-depth exploratory data analysis across 5 years of daily retail sales transactions (913,000 observations across 10 stores and 50 unique items).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
%matplotlib inline

### 1. Data Ingestion & Quality Audit

In [ ]:
df = pd.read_csv('../data/processed/train_clean.csv', parse_dates=['date'])
print(f"Total Observations: {len(df):,}")
print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
print(f"Stores: {df['store'].nunique()} | Items: {df['item'].nunique()}")
print(f"Missing Values: {df.isna().sum().sum()}")
df.head()

### 2. Macro Sales Trajectory (Aggregated Daily & Monthly)

In [ ]:
daily_total = df.groupby('date')['sales'].sum()
monthly_total = daily_total.resample('ME').sum()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_total.plot(ax=ax, color='#06B6D4', lw=2.5)
ax.set_title('Chain-wide Monthly Retail Sales (2013 - 2017)', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Units Sold')
plt.tight_layout()
plt.show()

### 3. Day-of-Week Seasonality (Weekend Surge)

In [ ]:
df['dow'] = df['date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_avg = df.groupby('dow')['sales'].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(10, 4))
dow_avg.plot(kind='bar', ax=ax, color='#3B82F6', edgecolor='#1E293B')
ax.set_title('Average Daily Sales by Day of Week', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Units / Store-Item')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4. Classical Seasonal Decomposition & Stationarity

In [ ]:
# Filter to Store 1, Item 1
s1i1 = df[(df['store'] == 1) & (df['item'] == 1)].set_index('date')['sales'].asfreq('D')
decomp = seasonal_decompose(s1i1, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
decomp.observed.plot(ax=axes[0], color='#64748B', title='Observed Sales')
decomp.trend.plot(ax=axes[1], color='#06B6D4', title='Extracted Trend Component')
decomp.seasonal.plot(ax=axes[2], color='#10B981', title='7-Day Seasonal Component')
decomp.resid.plot(ax=axes[3], color='#8B5CF6', title='Residual Irregular Noise')
plt.tight_layout()
plt.show()

# Augmented Dickey-Fuller Test
adf_res = adfuller(s1i1.dropna())
print(f"ADF Statistic: {adf_res[0]:.4f}")
print(f"p-value: {adf_res[1]:.4f} ({'Stationary' if adf_res[1] < 0.05 else 'Non-Stationary'})")